# Bank account management

This lab models bank accounts as objects. A procedural version, made of dictionaries and functions, is given: read it and run it, then rewrite it as objects, exercise after exercise. Each exercise builds on the previous one: if you get stuck, run its solution before moving on to the next. The first exercise, the class diagram, and the last step, the operations history, are optional: you can skip them. Amounts are whole numbers of euros.

## Specifications

- There are four types of accounts:
  - the current account, which allows an overdraft of at most €500;
  - the joint account, a current account with two holders;
  - the savings account, whose balance can never be negative;
  - the Livret A, a savings account whose balance can't exceed its ceiling of €22,950.
- Anyone can deposit money into an account; only one of its holders can withdraw from it, or transfer from it to another account.
- A transfer only happens if both accounts accept it; otherwise, neither balance changes.
- A customer has at most one account of each type. A joint account counts for both its holders.
- A refused operation raises an exception that says why.

## The Procedural Version

The following code follows these rules with dictionaries and functions, except the last rule about customers, which it ignores. Run it, then the next cell, which plays a scenario.

In [ ]:
from typing import Any

OVERDRAFT_LIMIT = 500
LIVRET_A_CEILING = 22_950


def create_account(account_type: str, *holders: str) -> dict[str, Any]:
  return {"type": account_type, "holders": holders, "balance": 0}


def check_balance(account: dict[str, Any], new_balance: int) -> None:
  if account["type"] in ("current", "joint") and new_balance < -OVERDRAFT_LIMIT:
    raise ValueError(f"Overdraft exceeded: the balance would become €{new_balance}")
  if account["type"] in ("savings", "livret A") and new_balance < 0:
    raise ValueError(f"Insufficient funds: the balance would become €{new_balance}")
  if account["type"] == "livret A" and new_balance > LIVRET_A_CEILING:
    raise ValueError(f"Ceiling exceeded: the balance would become €{new_balance}")


def check_holder(account: dict[str, Any], requester: str) -> None:
  if requester not in account["holders"]:
    raise ValueError(f"{requester} isn't a holder of this account")


def deposit(account: dict[str, Any], amount: int) -> None:
  check_balance(account, account["balance"] + amount)
  account["balance"] += amount


def withdraw(account: dict[str, Any], requester: str, amount: int) -> None:
  check_holder(account, requester)
  check_balance(account, account["balance"] - amount)
  account["balance"] -= amount


def transfer(
  source: dict[str, Any], requester: str, target: dict[str, Any], amount: int
) -> None:
  check_holder(source, requester)
  check_balance(source, source["balance"] - amount)
  check_balance(target, target["balance"] + amount)
  source["balance"] -= amount
  target["balance"] += amount

In [ ]:
current = create_account("current", "Isabelle")
savings = create_account("savings", "Jean")
livret = create_account("livret A", "Jean")
joint = create_account("joint", "Isabelle", "Jean")
deposit(current, 1300)
deposit(savings, 1000)
deposit(livret, 1200)
deposit(joint, 6500)
transfer(current, "Isabelle", joint, 500)
withdraw(livret, "Jean", 60)
print(current["balance"], savings["balance"], livret["balance"], joint["balance"])

try:
  withdraw(savings, "Jean", 2000)
except ValueError as e:
  print(f"Refused: {e}")

savings["balance"] = -1_000_000  # Nothing prevents it
print(savings)

This code works, but:

- each new account type adds `if`s to `check_balance`, and a misspelled type (`"Livret A"`) is subject to no rule at all;
- nothing prevents modifying a balance directly, as the last line does;
- no amount is checked: a €-100 deposit is a withdrawal anyone can make, and a €-100 withdrawal credits the account;
- every error is a `ValueError`: the caller can't tell a refusal the rules provide for from a programming error, nor an exceeded overdraft from a reached ceiling;
- nothing links a customer to their accounts, and the one-account-of-each-type rule isn't enforced.

The following exercises rewrite this code as objects, which settle each of these points.

## The Class Diagram (Optional)

Before writing the code, draw the class diagram of the specifications: the account types and their inheritance relationships, the customer and their accounts, with the attributes and methods of each class. Come back to it at the end of the lab, to compare it with the code you will have written.

Write it in Mermaid in the next text cell (double-click it to edit it): a ````mermaid` code block that starts with the line `classDiagram`. JupyterLab (4.1 and later) and GitHub display this block as a diagram; if your environment doesn't, the diagram's text stays readable as it is. A pen and paper will do too.

**Useful documentation**

- https://mermaid.js.org/syntax/classDiagram.html

*Your diagram here (double-click to edit this cell)*

### Additional hints

A class is written `class Name { ... }`, with one member per line, preceded by its visibility (the course, "Class Diagram: Notation"): `+` public, `#` protected (`_balance`). A method has parentheses, followed by its return type; a `*` after it marks it abstract, a `$` marks a class member, and `<<abstract>>`, on the first line, an abstract class. Generic types are written with tildes: `tuple~str~`. A relationship links two classes: `Parent <|-- Child` for inheritance, `Whole o-- Part` for an aggregation, `Whole *-- Part` for a composition, with the multiplicities in quotes and the relationship's name after `:`. A line starting with `%%` is a comment. A joint account has two holders, and a customer at most one account of each type: hence the multiplicities below.

```mermaid
classDiagram
  class Account {
    <<abstract>>
    +tuple~str~ holders
    #int _balance
    +balance() int
    #_check_balance(new_balance)*
    +deposit(amount)
    %% the other methods
  }
  class CurrentAccount {
    +int overdraft_limit$
    #_check_balance(new_balance)
  }
  class Customer {
    +str name
    %% its attributes and methods
  }
  Account <|-- CurrentAccount
  Customer "1..2" o-- "0..4" Account : accounts
  %% the other accounts, and their relations
```

### Solution

The diagram of the classes the lab builds, including the bank and the history of the last steps. The exceptions, a separate hierarchy, aren't shown.

```mermaid
classDiagram
  class Account {
    <<abstract>>
    +tuple~str~ holders
    #int _balance
    +list~Operation~ history
    +balance() int
    #_check_balance(new_balance)*
    #_check_amount(amount)$
    #_check_holder(requester)
    +deposit(amount)
    +withdraw(requester, amount)
    +transfer(requester, target, amount)
  }
  class CurrentAccount {
    +str label$
    +int overdraft_limit$
    #_check_balance(new_balance)
  }
  class JointAccount {
    +str label$
  }
  class SavingsAccount {
    +str label$
    #_check_balance(new_balance)
  }
  class LivretA {
    +str label$
    +int ceiling$
    #_check_balance(new_balance)
  }
  class Customer {
    +str name
    +dict accounts
    +add(account)
    +net_worth() int
  }
  class Bank {
    +dict TYPES$
    #list~Account~ _accounts
    +open(account_type, customers) Account
    +#95;#95;len#95;#95;() int
    +#95;#95;iter#95;#95;() Iterator
  }
  class Operation {
    <<dataclass>>
    +str kind
    +int amount
    +int balance
  }
  Account <|-- CurrentAccount
  CurrentAccount <|-- JointAccount
  Account <|-- SavingsAccount
  SavingsAccount <|-- LivretA
  Customer "1..2" o-- "0..4" Account : accounts
  Bank "1" *-- "*" Account : _accounts
  Account "1" *-- "*" Operation : history
```

An account exists without its customer, and a joint account belongs to two customers: the customer aggregates their accounts. The bank, on the other hand, creates the accounts and keeps them (composition), as each account does its history. In `#95;#95;len#95;#95;`, `#95;` is the code of the `_` character: without it, Mermaid would show `__len__` in bold.

## Business Exceptions

A refused operation raises a business exception, of a class that inherits from `BusinessError`. The next cell defines this hierarchy: run it. The `Error` suffix of their names follows the standard library's usage (`ValueError`, `KeyError`).

It lacks `InsufficientFundsError`, raised when the balance would become negative or exceed the allowed overdraft: write it. Unlike the others, it carries data. It is created with the account's balance and the requested amount, `InsufficientFundsError(balance, amount)`; it stores them in its `balance` and `amount` attributes, and builds its message from them, for example `Insufficient funds: €2000 requested, for a balance of €1000`.

Check that an `except BusinessError` catches it, and print its message and its `balance` attribute.

In [ ]:
class BusinessError(Exception):
  """Operation refused by the bank's rules."""


class CeilingExceededError(BusinessError):
  """The balance would exceed the account's ceiling."""


class AccessDeniedError(BusinessError):
  """The requester isn't a holder of the account."""


class DuplicateAccountError(BusinessError):
  """The customer already has an account of this type."""

In [ ]:
# Your code here

### Additional hints

`InsufficientFundsError` inherits from `BusinessError`. Its `__init__` stores its two parameters in `self`, then calls `super().__init__(message)`: that is the message `Exception` keeps, and `str(e)` returns. An `except` catches the exception of the class it names and those of all its subclasses, and `e` gives access to its attributes.

```python
class InsufficientFundsError(BusinessError):
  """The balance would become negative, or exceed the allowed overdraft."""

  def __init__(self, balance: int, amount: int) -> None:
    self.balance = balance
    self.amount = ...
    super().__init__(f"Insufficient funds: €{...} requested, for a balance of €{...}")


try:
  raise InsufficientFundsError(1000, 2000)
except BusinessError as e:
  print(f"Refused: {e} (balance: €{e.balance})")
```

### Solution

In [ ]:
class InsufficientFundsError(BusinessError):
  """The balance would become negative, or exceed the allowed overdraft."""

  def __init__(self, balance: int, amount: int) -> None:
    self.balance = balance
    self.amount = amount
    super().__init__(
      f"Insufficient funds: €{amount} requested, for a balance of €{balance}"
    )


try:
  raise InsufficientFundsError(1000, 2000)
except BusinessError as e:
  print(f"Refused: {type(e).__name__}: {e} (balance: €{e.balance})")

## The `Account` Abstract Class

Write an abstract class `Account`, the base class of every account:

- `__init__` receives the holder's name, stores the tuple of holders (`("Isabelle",)`) in the `holders` attribute, and a balance of 0 in the `_balance` attribute;
- the `balance` property gives the balance, read-only;
- the abstract method `_check_balance(new_balance)` raises a `BusinessError` if the account can't have this balance;
- the `_check_amount(amount)` method raises a `ValueError` if the amount isn't strictly positive: a zero or negative amount isn't a refusal the rules provide for, but a programming error of the caller;
- `deposit(amount)`, `withdraw(requester, amount)` and `transfer(requester, target, amount)` perform the operations of the specifications: they check the amount, and raise an `AccessDeniedError` when the requester isn't a holder. `transfer` credits the target by calling its `deposit` method, without touching its `_balance`;
- `__repr__` prints, for example, `CurrentAccount('Isabelle', balance=800)`.

Check that creating an `Account` raises a `TypeError`: the operations will be tested with the subclasses of the next exercise.

In [ ]:
# Your code here

### Additional hints

A property without a setter is read-only: `account.balance = 0` raises an `AttributeError`, while the class's methods modify `self._balance`. `_check_amount` uses neither the instance nor the class: it is a static method. The operations check everything before modifying anything: `transfer` checks the requester, the amount and the source's balance, then calls `target.deposit(amount)`, which makes the target's checks and raises before modifying its balance; only then does it debit the source. An object doesn't modify another's protected attribute: it sends it a message, by calling one of its methods. Each subclass will define its own rule in `_check_balance`, which `Account`'s operations call without knowing which type of account it is. A `_check_holder(requester)` method avoids writing the requester check twice. For `__repr__`, `", ".join(repr(h) for h in self.holders)` gives `'Isabelle', 'Jean'`.

```python
from abc import ABC, abstractmethod


class Account(ABC):
  def __init__(self, holder: str) -> None:
    self.holders = (holder,)
    self._balance = 0

  @property
  def balance(self) -> int:
    return ...

  @abstractmethod
  def _check_balance(self, new_balance: int) -> None: ...

  @staticmethod
  def _check_amount(amount: int) -> None:
    if ...:
      raise ValueError(f"The amount must be strictly positive, not {amount}")

  def _check_holder(self, requester: str) -> None:
    if ...:
      raise AccessDeniedError(f"{requester} isn't a holder of this account")

  def deposit(self, amount: int) -> None:
    self._check_amount(amount)
    self._check_balance(self._balance + amount)
    self._balance += amount

  def withdraw(self, requester: str, amount: int) -> None: ...

  def transfer(self, requester: str, target: "Account", amount: int) -> None:
    self._check_holder(requester)
    self._check_amount(amount)
    self._check_balance(...)
    target.deposit(...)
    ...

  def __repr__(self) -> str:
    holders = ", ".join(repr(h) for h in self.holders)
    return f"{type(self).__name__}({holders}, balance={...})"


try:
  Account("Isabelle")
except TypeError as e:
  print(e)
```

### Solution

In [ ]:
from abc import ABC, abstractmethod


class Account(ABC):
  def __init__(self, holder: str) -> None:
    self.holders = (holder,)
    self._balance = 0

  @property
  def balance(self) -> int:
    return self._balance

  @abstractmethod
  def _check_balance(self, new_balance: int) -> None:
    """Raise a BusinessError if the account can't have this balance."""

  @staticmethod
  def _check_amount(amount: int) -> None:
    if amount <= 0:
      raise ValueError(f"The amount must be strictly positive, not {amount}")

  def _check_holder(self, requester: str) -> None:
    if requester not in self.holders:
      raise AccessDeniedError(f"{requester} isn't a holder of this account")

  def deposit(self, amount: int) -> None:
    self._check_amount(amount)
    self._check_balance(self._balance + amount)
    self._balance += amount

  def withdraw(self, requester: str, amount: int) -> None:
    self._check_holder(requester)
    self._check_amount(amount)
    self._check_balance(self._balance - amount)
    self._balance -= amount

  def transfer(self, requester: str, target: "Account", amount: int) -> None:
    self._check_holder(requester)
    self._check_amount(amount)
    self._check_balance(self._balance - amount)
    target.deposit(amount)  # Checks the target, and raises before modifying it
    self._balance -= amount

  def __repr__(self) -> str:
    holders = ", ".join(repr(h) for h in self.holders)
    return f"{type(self).__name__}({holders}, balance={self._balance})"


try:
  Account("Isabelle")
except TypeError as e:
  print(f"Refused: {e}")

## Account Types

Write the four account types of the specifications, subclasses of `Account`:

- `CurrentAccount`, whose class attribute `overdraft_limit` is 500;
- `JointAccount`, a subclass of `CurrentAccount`, created with its two holders: `JointAccount("Isabelle", "Jean")`;
- `SavingsAccount`, whose balance can't be negative;
- `LivretA`, a subclass of `SavingsAccount`, whose class attribute `ceiling` is 22,950.

Each one also has a class attribute `label` (`"current account"`, `"joint account"`…), which the next exercise will use. A negative balance beyond the allowed overdraft raises an `InsufficientFundsError`, created with the account's current balance and the requested amount, and a balance above the ceiling a `CeilingExceededError`. Each redefinition of `_check_balance` has the `@override` decorator.

Then test them with the check cell that follows the solution.

In [ ]:
# Your code here

### Additional hints

Each class only defines what sets it apart: its rule in `_check_balance`, its class attributes, and, for `JointAccount`, an `__init__` that calls `super().__init__(holder)` then adds the co-holder to the `holders`. `JointAccount` inherits `CurrentAccount`'s rule. `LivretA`'s `_check_balance` first calls `SavingsAccount`'s with `super()`, then checks the ceiling. In a method, `self.overdraft_limit` reads the class attribute. `_check_balance` receives the new balance: the requested amount is the gap between the current balance and it, `self._balance - new_balance`. `@override`, from the `typing` module (Python 3.12), makes a type checker report a method that redefines nothing, for example because of a typo in its name.

```python
from typing import override


class CurrentAccount(Account):
  label = "current account"
  overdraft_limit = 500

  @override
  def _check_balance(self, new_balance: int) -> None:
    if new_balance < ...:
      raise InsufficientFundsError(self._balance, ...)


class JointAccount(CurrentAccount):
  label = "joint account"

  def __init__(self, holder: str, co_holder: str) -> None:
    super().__init__(holder)
    self.holders = ...


class SavingsAccount(Account): ...


class LivretA(SavingsAccount):
  label = "Livret A"
  ceiling = 22_950

  @override
  def _check_balance(self, new_balance: int) -> None:
    super()._check_balance(new_balance)
    ...
```

### Solution

In [ ]:
from typing import override


class CurrentAccount(Account):
  label = "current account"
  overdraft_limit = 500

  @override
  def _check_balance(self, new_balance: int) -> None:
    if new_balance < -self.overdraft_limit:
      raise InsufficientFundsError(self._balance, self._balance - new_balance)


class JointAccount(CurrentAccount):
  label = "joint account"

  def __init__(self, holder: str, co_holder: str) -> None:
    super().__init__(holder)
    self.holders = (holder, co_holder)


class SavingsAccount(Account):
  label = "savings account"

  @override
  def _check_balance(self, new_balance: int) -> None:
    if new_balance < 0:
      raise InsufficientFundsError(self._balance, self._balance - new_balance)


class LivretA(SavingsAccount):
  label = "Livret A"
  ceiling = 22_950

  @override
  def _check_balance(self, new_balance: int) -> None:
    super()._check_balance(new_balance)
    if new_balance > self.ceiling:
      raise CeilingExceededError(
        f"Ceiling of €{self.ceiling} exceeded: the balance would become €{new_balance}"
      )

### Check

The next cell replays the procedural version's scenario with your classes, then attempts operations the rules refuse. The `attempt` function runs an operation and prints whether it's accepted, or the business exception that refuses it; of an `InsufficientFundsError`, it prints the `amount` and `balance` attributes. Finally, a negative deposit raises a `ValueError`, which `attempt` lets through: it isn't a refusal the rules provide for, but a programming error.

Expected results: balances of €800, €1,000, €1,140 and €7,000; the 4th operation accepted, the others refused (€2,000 requested for a balance of €1,000, `CeilingExceededError`, `AccessDeniedError`, then €200 requested for a balance of -€400: the overdraft would exceed €500); at the end, -€400 on the current account and still €7,000 on the joint account, which the refused transfer didn't touch; an `AttributeError` when the balance is assigned, and a `ValueError` for the negative deposit.

In [ ]:
from collections.abc import Callable


def attempt(description: str, operation: Callable[[], object]) -> None:
  try:
    operation()
  except InsufficientFundsError as e:
    print(
      f"{description}: refused, €{e.amount} requested for a balance of €{e.balance}"
    )
  except BusinessError as e:
    print(f"{description}: refused ({type(e).__name__}: {e})")
  else:
    print(f"{description}: accepted")


current = CurrentAccount("Isabelle")
savings = SavingsAccount("Jean")
livret = LivretA("Jean")
joint = JointAccount("Isabelle", "Jean")
current.deposit(1300)
savings.deposit(1000)
livret.deposit(1200)
joint.deposit(6500)
current.transfer("Isabelle", joint, 500)
livret.withdraw("Jean", 60)
print(current, savings, livret, joint, sep="\n")

attempt("Jean withdraws €2000 from his savings", lambda: savings.withdraw("Jean", 2000))
attempt("Jean deposits €22,000 into his Livret A", lambda: livret.deposit(22_000))
attempt("Jean withdraws €100 from Isabelle's", lambda: current.withdraw("Jean", 100))
attempt("Isabelle withdraws €1200", lambda: current.withdraw("Isabelle", 1200))
attempt(
  "Isabelle transfers €200 to joint", lambda: current.transfer("Isabelle", joint, 200)
)
print(current, joint, sep="\n")

try:
  current.balance = 1_000_000
except AttributeError as e:
  print(f"Refused: {e}")

try:
  current.deposit(-100)
except ValueError as e:
  print(f"Programming error: {e}")

## Customers

Write a `Customer` class, created with the customer's name, which associates a customer with their accounts:

- the `accounts` attribute is a dictionary, empty at first, whose keys are the account types (the classes themselves, such as `CurrentAccount`) and whose values are the customer's accounts;
- `add(account)` adds an account to the customer, raising an `AccessDeniedError` if the customer isn't one of its holders, and a `DuplicateAccountError` if they already have an account of this type;
- the `net_worth` property gives the sum of their accounts' balances;
- `__repr__` prints the customer's name and the list of their accounts.

An account exists without its customer, and a joint account belongs to two customers: this is an aggregation, not a composition. Test your class with the check cell that follows the solution.

In [ ]:
# Your code here

### Additional hints

`type(account)` returns an account's class, which serves as the key: `JointAccount` and `CurrentAccount` are two different keys, even though one inherits from the other. `in` tests whether a key is in a dictionary, and `sum` adds up the balances a generator expression over `self.accounts.values()` produces. The `DuplicateAccountError` message can use the account's `label` class attribute: `Isabelle already has a current account`.

```python
class Customer:
  def __init__(self, name: str) -> None:
    self.name = name
    self.accounts: dict[type[Account], Account] = {}

  def add(self, account: Account) -> None:
    if self.name not in account.holders:
      raise AccessDeniedError(...)
    if type(account) in self.accounts:
      raise DuplicateAccountError(f"{self.name} already has a {account.label}")
    self.accounts[...] = account

  @property
  def net_worth(self) -> int:
    return sum(...)

  def __repr__(self) -> str:
    return f"Customer({self.name!r}, accounts={list(...)})"
```

### Solution

In [ ]:
class Customer:
  def __init__(self, name: str) -> None:
    self.name = name
    self.accounts: dict[type[Account], Account] = {}

  def add(self, account: Account) -> None:
    if self.name not in account.holders:
      raise AccessDeniedError(f"{self.name} isn't a holder of {account!r}")
    if type(account) in self.accounts:
      raise DuplicateAccountError(f"{self.name} already has a {account.label}")
    self.accounts[type(account)] = account

  @property
  def net_worth(self) -> int:
    return sum(account.balance for account in self.accounts.values())

  def __repr__(self) -> str:
    return f"Customer({self.name!r}, accounts={list(self.accounts.values())})"

### Check

Expected results: Isabelle has the current account and the joint account, Jean the savings account, the Livret A and the joint account; their net worths are €6,600 and €9,140 (the joint account counts for both); Isabelle's second current account is refused (`DuplicateAccountError`), as is Jean's Livret A (`AccessDeniedError`), but not a Livret A in her name.

In [ ]:
isabelle = Customer("Isabelle")
jean = Customer("Jean")
isabelle.add(current)
isabelle.add(joint)
jean.add(savings)
jean.add(livret)
jean.add(joint)
print(isabelle, jean, sep="\n")
print(isabelle.net_worth, jean.net_worth)

attempt(
  "Isabelle opens a second current account",
  lambda: isabelle.add(CurrentAccount("Isabelle")),
)
attempt("Isabelle adds Jean's Livret A", lambda: isabelle.add(livret))
attempt("Isabelle opens a Livret A", lambda: isabelle.add(LivretA("Isabelle")))
print(isabelle.accounts[LivretA])

## The Bank

A bank opens its customers' accounts and keeps them: it creates them itself, this is a composition. Write a `Bank` class:

- its `TYPES` class attribute maps each account type name (`"current"`, `"joint"`, `"savings"` and `"livret A"`) to its class: this is the course's factory, a dictionary of classes;
- `open(account_type, *customers)` creates an account of the named type, in the name of the given customer (of both, for a joint account), adds it to each customer's accounts and to the bank's, and returns it. If one of the customers already has an account of this type, it raises a `DuplicateAccountError` without creating anything; for an unknown type, a `ValueError`;
- `len(bank)` gives the number of open accounts, and `for account in bank` goes through them: these are the `__len__` and `__iter__` special methods.

Test your class with the check cell that follows the solution.

In [ ]:
# Your code here

### Additional hints

`TYPES` is written in the class body, like `EXPORTERS` in the course's factory. `self.TYPES[account_type]` gives a class, which is called like any class to create the account: `cls("Claire")`. With `*customers`, `open` receives one or two customers, whose names are the account's holders: `cls(*(customer.name for customer in customers))` passes them all. `Customer.add` already checks the one-account-per-type rule, but check every customer before creating anything: if the second holder of a joint account already has one, the first must not receive a new one. `__len__` returns an integer, and `__iter__` an iterator, which `iter(self._accounts)` gives.

```python
from collections.abc import Iterator


class Bank:
  TYPES: dict[str, type[Account]] = {
    "current": CurrentAccount,
    "joint": JointAccount,
    "savings": ...,
    "livret A": ...,
  }

  def __init__(self) -> None:
    self._accounts: list[Account] = []

  def open(self, account_type: str, *customers: Customer) -> Account:
    if account_type not in self.TYPES:
      raise ValueError(f"Unknown account type: {account_type!r}")
    cls = self.TYPES[account_type]
    for customer in customers:
      if ...:
        raise DuplicateAccountError(f"{customer.name} already has a {cls.label}")
    account = cls(*(customer.name for customer in customers))
    ...
    return account

  def __len__(self) -> int:
    return ...

  def __iter__(self) -> Iterator[Account]:
    return ...
```

### Solution

In [ ]:
from collections.abc import Iterator


class Bank:
  TYPES: dict[str, type[Account]] = {
    "current": CurrentAccount,
    "joint": JointAccount,
    "savings": SavingsAccount,
    "livret A": LivretA,
  }

  def __init__(self) -> None:
    self._accounts: list[Account] = []

  def open(self, account_type: str, *customers: Customer) -> Account:
    if account_type not in self.TYPES:
      raise ValueError(f"Unknown account type: {account_type!r}")
    cls = self.TYPES[account_type]
    for customer in customers:  # Check everything before creating anything
      if cls in customer.accounts:
        raise DuplicateAccountError(f"{customer.name} already has a {cls.label}")
    account = cls(*(customer.name for customer in customers))
    for customer in customers:
      customer.add(account)
    self._accounts.append(account)
    return account

  def __len__(self) -> int:
    return len(self._accounts)

  def __iter__(self) -> Iterator[Account]:
    return iter(self._accounts)

### Check

The bank opens the accounts of three new customers, Claire, Marc and Léa, then more are attempted.

Expected results: Claire's second current account refused (`DuplicateAccountError`), as is Léa and Marc's joint account, since he already has one; the misspelled "Livret A" type refused with a `ValueError`; the bank has 4 accounts, including Marc's savings, credited with €500; and Léa has no account, although the refusal came from Marc.

In [ ]:
bank = Bank()
claire = Customer("Claire")
marc = Customer("Marc")
lea = Customer("Léa")
bank.open("current", claire)
bank.open("livret A", claire)
bank.open("joint", claire, marc)
bank.open("savings", marc).deposit(500)

attempt("Claire opens a second current account", lambda: bank.open("current", claire))
attempt("Léa opens a joint account with Marc", lambda: bank.open("joint", lea, marc))
try:
  bank.open("Livret A", lea)
except ValueError as e:
  print(f"Programming error: {e}")

print(len(bank))
for account in bank:
  print(account)
print(claire, marc, lea, sep="\n")

## The Operations History (Optional)

This last step is optional: skip it if you're short of time. Each account keeps the history of the operations it accepted:

- a frozen (`frozen=True`) `Operation` dataclass describes an operation by its kind (`"deposit"`, `"withdrawal"` or `"transfer"`), its amount and the account's balance after the operation;
- an account's `history` attribute is a list of `Operation`s, empty when it is created, to which `deposit`, `withdraw` and `transfer` add the operation they just made. A transfer is recorded as a transfer on the debited account, and as a deposit on the credited account;
- a refused operation appears in no history.

This requires modifying `Account`, then redefining its four subclasses, even without changing them: a class inherits from the parent class that existed when it was defined, not from the one that bears that name today. Finally, check that an `Operation` can't be modified.

In [ ]:
# Your code here

### Additional hints

`@dataclass(frozen=True)`, from the `dataclasses` module, writes `__init__`, `__repr__` and `__eq__` from the annotated fields, and forbids modifying them after creation: an assignment raises a `FrozenInstanceError`, a subclass of `AttributeError`. Each operation creates an `Operation` after modifying the balance, and adds it to the history with `append`. In `transfer`, `target.deposit(amount)` already records the deposit in the target's history. Expected from the check: on the current account, a €1,000 deposit, a €300 transfer and a €50 withdrawal, for a final balance of €650; on the Livret A, a €300 deposit.

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Operation:
  kind: str
  ...


class Account(ABC):
  def __init__(self, holder: str) -> None:
    self.holders = (holder,)
    self._balance = 0
    self.history: list[Operation] = []

  ...  # balance, _check_balance, _check_amount and _check_holder unchanged

  def deposit(self, amount: int) -> None:
    self._check_amount(amount)
    self._check_balance(self._balance + amount)
    self._balance += amount
    self.history.append(Operation("deposit", amount, self._balance))

  ...  # withdraw and transfer likewise, then __repr__


...  # CurrentAccount, JointAccount, SavingsAccount and LivretA, copied as they are
```

### Solution

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import override


@dataclass(frozen=True)
class Operation:
  kind: str
  amount: int
  balance: int  # The account's balance after the operation


class Account(ABC):
  def __init__(self, holder: str) -> None:
    self.holders = (holder,)
    self._balance = 0
    self.history: list[Operation] = []

  @property
  def balance(self) -> int:
    return self._balance

  @abstractmethod
  def _check_balance(self, new_balance: int) -> None:
    """Raise a BusinessError if the account can't have this balance."""

  @staticmethod
  def _check_amount(amount: int) -> None:
    if amount <= 0:
      raise ValueError(f"The amount must be strictly positive, not {amount}")

  def _check_holder(self, requester: str) -> None:
    if requester not in self.holders:
      raise AccessDeniedError(f"{requester} isn't a holder of this account")

  def deposit(self, amount: int) -> None:
    self._check_amount(amount)
    self._check_balance(self._balance + amount)
    self._balance += amount
    self.history.append(Operation("deposit", amount, self._balance))

  def withdraw(self, requester: str, amount: int) -> None:
    self._check_holder(requester)
    self._check_amount(amount)
    self._check_balance(self._balance - amount)
    self._balance -= amount
    self.history.append(Operation("withdrawal", amount, self._balance))

  def transfer(self, requester: str, target: "Account", amount: int) -> None:
    self._check_holder(requester)
    self._check_amount(amount)
    self._check_balance(self._balance - amount)
    target.deposit(amount)  # Records the deposit in the target's history
    self._balance -= amount
    self.history.append(Operation("transfer", amount, self._balance))

  def __repr__(self) -> str:
    holders = ", ".join(repr(h) for h in self.holders)
    return f"{type(self).__name__}({holders}, balance={self._balance})"


# The subclasses, copied as they are, so that they inherit from the new Account
class CurrentAccount(Account):
  label = "current account"
  overdraft_limit = 500

  @override
  def _check_balance(self, new_balance: int) -> None:
    if new_balance < -self.overdraft_limit:
      raise InsufficientFundsError(self._balance, self._balance - new_balance)


class JointAccount(CurrentAccount):
  label = "joint account"

  def __init__(self, holder: str, co_holder: str) -> None:
    super().__init__(holder)
    self.holders = (holder, co_holder)


class SavingsAccount(Account):
  label = "savings account"

  @override
  def _check_balance(self, new_balance: int) -> None:
    if new_balance < 0:
      raise InsufficientFundsError(self._balance, self._balance - new_balance)


class LivretA(SavingsAccount):
  label = "Livret A"
  ceiling = 22_950

  @override
  def _check_balance(self, new_balance: int) -> None:
    super()._check_balance(new_balance)
    if new_balance > self.ceiling:
      raise CeilingExceededError(
        f"Ceiling of €{self.ceiling} exceeded: the balance would become €{new_balance}"
      )

### Check

Expected results: the €2,000 transfer refused (insufficient funds); three operations on the current account, from `Operation(kind='deposit', amount=1000, balance=1000)` to a €50 withdrawal that leaves a balance of €650; a single €300 deposit on the Livret A; and a `FrozenInstanceError` when an operation is modified.

In [ ]:
current = CurrentAccount("Isabelle")
livret = LivretA("Isabelle")
current.deposit(1000)
current.transfer("Isabelle", livret, 300)
current.withdraw("Isabelle", 50)
attempt(
  "Isabelle transfers €2000 to her Livret A",
  lambda: current.transfer("Isabelle", livret, 2000),
)
for operation in current.history:
  print(operation)
print(livret.history)

try:
  current.history[0].amount = 1_000_000
except AttributeError as e:
  print(f"Refused: {type(e).__name__}: {e}")